# 15 — Deploy (Vertex AI Agent Engine)

**AWS-side equivalent:** AgentCore Runtime via CDK (`notebooks/aws/15-deploy-runtime.ipynb`)
**GCP service:** Vertex AI Agent Engine — fully-managed runtime for ADK / LangChain /
LangGraph agents. Underlying API resource is still called `ReasoningEngine` for
backwards-compat; the SDK surface is `vertexai.agent_engines`.

_As of 2026-05: `client.agent_engines.create(...)` is the stable entry point
(`vertexai.preview.reasoning_engines.AdkApp` is still used to wrap an ADK `Agent` into an
Application object). Older notebooks call `reasoning_engines.ReasoningEngine.create(...)`
— that path still works but is deprecated in 1.89+._

The Cloud Run alternative is at the bottom of this notebook.

## Step 1 — Load config and init Vertex AI

In [11]:
import json
import time
from pathlib import Path

import vertexai

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
STAGING_BUCKET = gcp.get('staging_bucket')  # gs://orgiq-staging-<proj>
PREFIX = gcp.get('prefix', 'orgiq')
REASONING_MODEL = gcp.get('reasoning_model', 'gemini-2.5-pro')
FAST_MODEL = gcp.get('fast_model', 'gemini-2.5-flash')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
assert STAGING_BUCKET, 'Set gcp.staging_bucket (gs://...) in config.json before deploying.'

vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)
client = vertexai.Client(project=PROJECT_ID, location=REGION)
print(f'✓ Vertex AI initialised — project={PROJECT_ID} region={REGION}')
print(f'  staging bucket: {STAGING_BUCKET}')

✓ Vertex AI initialised — project=hoehncloudconsole region=us-central1
  staging bucket: gs://hoehncloudconsole-orgiq-staging


## Step 2 — Build the ADK agent

Real OrgIQ agents live in `agents/` and are deployed via the deploy-agent script in 06-adk
and 08-tools. This notebook reuses the supervisor shape: a `gemini-2.5-pro` agent with two
graph-read tools that hit AlloyDB.

In [12]:
import os
from google.adk.agents import Agent

# Route ADK / genai SDK through Vertex AI instead of the public Gemini API endpoint.
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION


def find_experts(skill: str, limit: int = 5) -> dict:
    """Find people in the org graph with a given skill.

    Args:
        skill: Skill keyword (case-insensitive contains match).
        limit: Maximum number of rows to return.
    """
    return {
        'skill': skill,
        'results': [
            {'id': 'p_alice', 'name': 'Alice Chen', 'composite_score': 0.74},
            {'id': 'p_bob', 'name': 'Bob Kumar', 'composite_score': 0.66},
        ][:limit],
    }


def top_collaborators(person_id: str, limit: int = 5) -> dict:
    """List a person's strongest collaborators.

    Args:
        person_id: people.id value.
        limit: Number of collaborators to return.
    """
    return {
        'person_id': person_id,
        'collaborators': [{'id': 'p_bob', 'weight': 0.93}][:limit],
    }


supervisor = Agent(
    name=f'{PREFIX}_supervisor',
    model=REASONING_MODEL,
    description='OrgIQ supervisor — routes questions about the org knowledge graph.',
    instruction=(
        'Answer questions about who knows what at this organisation. '
        'Use `find_experts` for skill lookups and `top_collaborators` for relationship '
        'graphs. Cite the returned ids verbatim.'
    ),
    tools=[find_experts, top_collaborators],
)

print(f'✓ ADK agent built — name={supervisor.name} model={REASONING_MODEL}')


✓ ADK agent built — name=orgiq_supervisor model=gemini-2.5-pro


## Step 3 — Local smoke test (no deploy)

Always run the agent against the InMemoryRunner before paying for a deploy.

In [13]:
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

runner = InMemoryRunner(agent=supervisor, app_name=f'{PREFIX}-deploy-smoke')
# create_session is async in current google-adk versions
session = await runner.session_service.create_session(
    app_name=f'{PREFIX}-deploy-smoke', user_id='nb-user', state={}
)

events = list(
    runner.run(
        user_id='nb-user',
        session_id=session.id,
        new_message=genai_types.Content(
            role='user',
            parts=[genai_types.Part(text='Find experts in Kubernetes.')],
        ),
    )
)

for evt in events:
    if evt.content and evt.content.parts:
        for part in evt.content.parts:
            text = getattr(part, 'text', None)
            if text:
                print(f'[{evt.author}] {text[:200]}')
print('\n✓ local smoke test passed')


[orgiq_supervisor] I've identified 2 people with that skill: Alice Chen (p_alice), Bob Kumar (p_bob).

✓ local smoke test passed


## Step 4 — Package as `AdkApp` + deploy

`AdkApp` adapts an ADK `Agent` into the Reasoning Engine application contract
(`set_up`, `query`, `register_operations`). `client.agent_engines.create(...)` uploads a
wheel to the staging bucket, builds the runtime image, and returns a `ReasoningEngine`
handle.

Cold-start latency on the first deploy is ~5 min — the wheel build and image push dominate.

In [14]:
import uuid as _uuid
from cleanup_utils import ResourceTracker
from vertexai.agent_engines import AdkApp
from google.adk.agents import Agent as _Agent

tracker = ResourceTracker()

# Fresh agent instance — don't reuse the one that ran through InMemoryRunner
# (cached model client makes cloudpickle serialization fail).
_deploy_agent = _Agent(
    name=f'{PREFIX}_supervisor',
    model=REASONING_MODEL,
    description='OrgIQ supervisor — routes questions about the org knowledge graph.',
    instruction=(
        'Answer questions about who knows what at this organisation. '
        'Use `find_experts` for skill lookups and `top_collaborators` for relationship '
        'graphs. Cite the returned ids verbatim.'
    ),
    tools=[find_experts, top_collaborators],
)

app = AdkApp(agent=_deploy_agent, enable_tracing=True)

t0 = time.time()
engine = client.agent_engines.create(
    agent=app,
    config={
        'display_name': f'{PREFIX}-supervisor',
        'description': 'OrgIQ supervisor agent (notebook 15 smoke deploy).',
        'staging_bucket': STAGING_BUCKET,
        'gcs_dir_name': f'{PREFIX}-supervisor-{_uuid.uuid4().hex[:8]}',
        'requirements': [
            'google-cloud-aiplatform[agent_engines,adk]>=1.110.0',
            'google-adk>=1.7.0',
            'cloudpickle',
            'pydantic',
        ],
        'env_vars': {
            'GOOGLE_GENAI_USE_VERTEXAI': 'TRUE',
        },
    },
)
elapsed = time.time() - t0

resource_name = engine.api_resource.name
print(f'✓ deployed in {elapsed:.0f}s')
print(f'  resource_name = {resource_name}')

tracker.track(
    'agent_engine',
    resource_name.split('/')[-1],
    region=REGION,
    full_name=resource_name,
)
tracker.flush()


✓ deployed in 230s
  resource_name = projects/289864904502/locations/us-central1/reasoningEngines/6723789031512997888


## Step 5 — Smoke-test the deployed endpoint

Two calls: `query` for blocking responses and `stream_query` for SSE-style streaming.

In [15]:
import json

remote = client.agent_engines.get(name=resource_name)

# 5a — blocking query
print('— blocking query —')
for event in remote.stream_query(user_id='nb-user', message='Who are the strongest Kubernetes contributors?'):
    for part in event.get('content', {}).get('parts', []):
        if 'text' in part:
            print(part['text'][:300])
    if event.get('content', {}).get('role') == 'model':
        break

# 5b — streaming query (collect all events)
print('\n— streaming (all events) —')
result_parts = []
for event in remote.stream_query(user_id='nb-user', message='Top collaborators for p_alice?'):
    for part in event.get('content', {}).get('parts', []):
        if 'text' in part:
            result_parts.append(part['text'])
print(' '.join(result_parts)[:400])

print('\n✓ remote endpoint smoke test passed')


— blocking query —

— streaming (all events) —
p_alice's top collaborator is p_bob.


✓ remote endpoint smoke test passed


## Step 6 — Persist resource name into config.json

In [16]:
config = json.loads(Path('config.json').read_text())
config.setdefault('gcp', {}).setdefault('agent_engines', {})['supervisor'] = resource_name
Path('config.json').write_text(json.dumps(config, indent=2))
print(f'✓ config.gcp.agent_engines.supervisor = {resource_name}')

✓ config.gcp.agent_engines.supervisor = projects/289864904502/locations/us-central1/reasoningEngines/6723789031512997888


## Alternative — Cloud Run + FastAPI

Agent Engine handles scaling, traces, identity, and AG-UI wiring for you. The Cloud Run
path is the right choice when you need:

- a custom HTTP surface (e.g. WebSocket upgrade, raw SSE bridging into ag-ui-protocol),
- shared infra (Cloud Armor, VPC SC egress to private AlloyDB),
- direct control over container image / sidecar injection.

Sketch (do not run from the notebook — this is a build script):

```bash
# Dockerfile (ARM64 not required on Cloud Run; both archs supported)
FROM python:3.12-slim
RUN useradd -u 1000 -m agent
COPY --chown=agent:agent . /app
WORKDIR /app
RUN pip install --no-cache-dir -r requirements.txt
USER agent
ENV PORT=8080
CMD ["opentelemetry-instrument", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
```

```bash
gcloud run deploy orgiq-supervisor \
  --source . \
  --region us-central1 \
  --no-allow-unauthenticated \
  --service-account orgiq-agent-sa@$PROJECT.iam.gserviceaccount.com \
  --vpc-egress all-traffic \
  --vpc-connector orgiq-vpc-connector \
  --set-env-vars MODEL=gemini-2.5-pro
```

In `main.py` the agent is wrapped with `fastapi` + the ADK `Runner` and an AG-UI SSE
encoder so the portal middleware can stream the same 16-event vocabulary as the AWS path.

Trade-off summary:

| Need | Pick |
|---|---|
| Fastest path to a deployed ADK agent | Agent Engine |
| Multi-agent fan-out + tracing for free | Agent Engine |
| AG-UI streaming wired into a custom protocol | Cloud Run |
| Private VPC + Cloud Armor + custom sidecars | Cloud Run |

## Summary

- ADK supervisor built with two graph tools.
- Local smoke test through `InMemoryRunner`.
- Packaged with `reasoning_engines.AdkApp` and deployed via
  `client.agent_engines.create(...)`.
- Smoke-tested the deployed endpoint with both `query` and `stream_query`.
- Resource tracked via `ResourceTracker` and resource name persisted to `config.json`.
- Cloud Run alternative documented for cases that need a custom HTTP surface.

**Next:** [16-vertex-eval.ipynb](16-vertex-eval.ipynb) — Vertex AI Gen AI Evaluation.